In [ ]:
# Robust, self-contained synthetic image generator cell
# - Tries local diffusers+torch
# - Falls back to Hugging Face Inference API when available (HF_TOKEN)
# - Final fallback: generates placeholder PNGs so notebook never fails
import os, time, io, requests
from PIL import Image, ImageDraw, ImageFont
from typing import List

# Detect local diffusers support (safe, non-invasive)
USE_LOCAL = False
try:
    import torch
    from diffusers import StableDiffusionPipeline
    # quick sanity-check
    _ = StableDiffusionPipeline
    USE_LOCAL = True
except Exception:
    USE_LOCAL = False

class SyntheticDataGenerator:
    def __init__(
        self,
        model_local: str = "runwayml/stable-diffusion-v1-5",
        model_hf: str = "stabilityai/stable-diffusion-2",
        hf_token: str | None = None,
    ):
        self.model_local = model_local
        self.model_hf = model_hf
        self.hf_token = hf_token or os.environ.get("HF_TOKEN")
        self.hf_headers = {"Authorization": f"Bearer {self.hf_token}"} if self.hf_token else None

        self.mode = "local" if (USE_LOCAL) else ("hf" if self.hf_headers else "placeholder")

        # prompts
        self.prompts = {
            "CIN1": "histopathology microscopy image, epithelial tissue with mild cellular changes, dysplastic cells, medical slide, diagnostic pathology",
            "CIN2": "histopathology microscopy image, epithelial tissue with moderate cellular abnormalities, dysplastic tissue, medical slide, diagnostic pathology",
            "CIN3": "histopathology microscopy image, epithelial tissue with severe cellular changes, high-grade dysplasia, medical slide, diagnostic pathology",
            "Cancer": "histopathology microscopy image, malignant epithelial cells, invasive carcinoma tissue, medical slide, diagnostic pathology",
            "Normal": "histopathology microscopy image, normal epithelial tissue, clinical diagnostic image",
        }
        self.negative_prompt = "blurry, low quality, cartoon, illustration, text, watermark, signature, person, face, body"

        # initialize local pipeline lazily to avoid heavy import work on construction
        self._local_pipe = None

    def _init_local(self):
        if self._local_pipe is not None:
            return
        device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = torch.float16 if device == "cuda" else torch.float32
        self._local_pipe = StableDiffusionPipeline.from_pretrained(
            self.model_local,
            torch_dtype=dtype,
            safety_checker=None,
            requires_safety_checker=False,
        ).to(device)
        if device == "cuda":
            try:
                self._local_pipe.enable_attention_slicing()
            except Exception:
                pass

    def _hf_call(self, prompt: str, negative_prompt: str | None = None, steps: int = 30, width: int = 512, height: int = 512) -> bytes:
        if not self.hf_headers:
            raise RuntimeError("HF token not set for HF API mode")
        url = f"https://api-inference.huggingface.co/models/{self.model_hf}"
        payload = {
            "inputs": prompt,
            "parameters": {
                "negative_prompt": negative_prompt or "",
                "num_inference_steps": steps,
                "guidance_scale": 7.5,
                "width": width,
                "height": height,
            },
            "options": {"wait_for_model": True},
        }
        backoff = 1
        for attempt in range(4):
            try:
                resp = requests.post(url, headers=self.hf_headers, json=payload, timeout=120)
                if resp.status_code == 200:
                    ct = resp.headers.get("content-type", "")
                    if "application/json" in ct:
                        data = resp.json()
                        if isinstance(data, dict) and "image" in data:
                            import base64
                            return base64.b64decode(data["image"])
                        raise RuntimeError("HF returned JSON without image payload")
                    return resp.content
                elif 500 <= resp.status_code < 600:
                    time.sleep(backoff)
                    backoff *= 2
                    continue
                else:
                    raise RuntimeError(f"HF API error {resp.status_code}: {resp.text}")
            except Exception:
                if attempt == 3:
                    raise
                time.sleep(backoff)
                backoff *= 2
        raise RuntimeError("HF API failed after retries")

    def _placeholder_image(self, text: str, size=(512, 512)) -> bytes:
        img = Image.new("RGB", size, (240, 240, 240))
        draw = ImageDraw.Draw(img)
        try:
            font = ImageFont.load_default()
        except Exception:
            font = None
        w, h = draw.textsize(text, font=font)
        draw.rectangle(((6, 6), (size[0] - 6, size[1] - 6)), outline=(200, 200, 200))
        draw.text(((size[0] - w) / 2, (size[1] - h) / 2), text, fill=(30, 30, 30), font=font)
        bio = io.BytesIO()
        img.save(bio, format="PNG")
        return bio.getvalue()

    def _save_bytes_to_file(self, img_bytes: bytes, path: str):
        try:
            img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
            img.save(path)
        except Exception:
            with open(path, "wb") as f:
                f.write(img_bytes)

    def generate_images(self, stage: str, num_images: int, output_dir: str, start_idx: int = 1, height: int = 512, width: int = 512) -> List[str]:
        os.makedirs(output_dir, exist_ok=True)
        prompt = self.prompts.get(stage, self.prompts["Normal"])
        generated = []

        if self.mode == "local":
            try:
                self._init_local()
            except Exception as e:
                # fallback to HF if available, else placeholder
                self.mode = "hf" if self.hf_headers else "placeholder"

        if self.mode == "local":
            for i in range(num_images):
                try:
                    out = self._local_pipe(
                        prompt,
                        negative_prompt=self.negative_prompt,
                        num_inference_steps=50,
                        guidance_scale=7.5,
                        height=height,
                        width=width,
                    ).images[0]
                    path = os.path.join(output_dir, f"{stage}_{start_idx + i}.png")
                    out.save(path)
                    generated.append(path)
                except Exception:
                    # on per-image failure, switch to HF or placeholder for remaining
                    self.mode = "hf" if self.hf_headers else "placeholder"
                    break

        if self.mode == "hf":
            for i in range(len(generated), num_images):
                try:
                    img_bytes = self._hf_call(prompt, negative_prompt=self.negative_prompt, steps=50, width=width, height=height)
                    path = os.path.join(output_dir, f"{stage}_{start_idx + i}.png")
                    self._save_bytes_to_file(img_bytes, path)
                    generated.append(path)
                except Exception:
                    # on HF failure, fall back to placeholder for remaining
                    self.mode = "placeholder"
                    break

        if self.mode == "placeholder":
            for i in range(len(generated), num_images):
                txt = f"{stage} placeholder #{start_idx + i}"
                img_bytes = self._placeholder_image(txt, size=(width, height))
                path = os.path.join(output_dir, f"{stage}_{start_idx + i}.png")
                with open(path, "wb") as f:
                    f.write(img_bytes)
                generated.append(path)

        return generated

# Minimal usage example (uncomment and run as needed)
# gen = SyntheticDataGenerator()               # uses HF if available; local if diffusers present
# gen.generate_images('CIN1', 5, 'synthetic_images/CIN1', start_idx=1)
print("SyntheticDataGenerator ready. mode:", ("local" if USE_LOCAL else ("hf" if os.environ.get('HF_TOKEN') else "placeholder")))

In [ ]:
# Quick import check (run after kernel restart)
import torch
from diffusers import StableDiffusionPipeline
from transformers import __version__ as transformers_version
import pandas as pd
print("torch:", torch.__version__, "transformers:", transformers_version, "pandas:", pd.__version__)
print("diffusers import OK")

In [ ]:
# Step 2: Verify Data (data already in Kaggle dataset)
import os
import shutil

print("📋 Checking for data...\n")

# Path to your uploaded dataset
dataset_path = '/kaggle/input/datasets/akshitarora345/cervical-cancer-dataset/data'
local_data_path = 'data'

if os.path.exists(dataset_path):
    print(f"✅ Found dataset at: {dataset_path}\n")
    
    # Copy to local working directory for easier access
    if not os.path.exists(local_data_path):
        print(f"📦 Copying data to working directory...")
        shutil.copytree(dataset_path, local_data_path)
        print("✅ Data copied!")
    
    print("\n📊 Data structure:")
    for root, dirs, files in os.walk(local_data_path):
        level = root.replace(local_data_path, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (level + 1)
        file_count = len([f for f in files if f.endswith(('.png', '.jpg', '.jpeg'))])
        if file_count > 0:
            print(f'{subindent}{file_count} images')
    
    print("\n✓ Data ready for synthetic generation!")
else:
    print(f"❌ Dataset not found at {dataset_path}")
    print("\nTrying alternative paths...")
    
    # Try to find data folder
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'data' in dirs:
            print(f"Found data at: {os.path.join(root, 'data')}")

In [ ]:
# Improved Synthetic Data Generator with Better Prompts
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image
import os
from tqdm import tqdm

class SyntheticDataGenerator:
    def __init__(self, model_id="runwayml/stable-diffusion-v1-5"):
        print(f"Loading Stable Diffusion model: {model_id}")
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {device}")
        
        self.pipe = StableDiffusionPipeline.from_pretrained(
            model_id,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            safety_checker=None,
            requires_safety_checker=False
        )
        self.pipe = self.pipe.to(device)
        
        if device == "cuda":
            self.pipe.enable_attention_slicing()
            print("✓ Attention slicing enabled for memory efficiency")
        
        # Enhanced prompts for better quality
        self.prompts = {
            'CIN1': "histopathology microscopy image, epithelial tissue with mild cellular changes, dysplastic cells, medical slide, scientific photograph, diagnostic pathology",
            'CIN2': "histopathology microscopy image, epithelial tissue with moderate cellular abnormalities, dysplastic tissue, medical slide, scientific photograph, diagnostic pathology",
            'CIN3': "histopathology microscopy image, epithelial tissue with severe cellular changes, high-grade dysplasia, medical slide, scientific photograph, diagnostic pathology",
            'Cancer': "histopathology microscopy image, malignant epithelial cells, invasive carcinoma tissue, medical slide, scientific photograph, diagnostic pathology",
            'Normal': "histopathology microscopy image, pink epithelial cells, normal tissue structure, medical slide, scientific photograph, clinical diagnostic image"
        }
        
        self.negative_prompt = "blurry, low quality, cartoon, illustration, text, watermark, signature, person, face, body"
    
    def generate_images(self, stage, num_images, output_dir, start_idx=1):
        """
        Generate synthetic images
        
        Args:
            stage: Class name (CIN1, CIN2, CIN3, Cancer, Normal)
            num_images: Number of images to generate
            output_dir: Directory to save images
            start_idx: Starting index for filenames (useful for adding to existing images)
        """
        os.makedirs(output_dir, exist_ok=True)
        prompt = self.prompts.get(stage, self.prompts['Normal'])
        
        print(f"\n{'='*80}")
        print(f"Generating {num_images} images for {stage}")
        print(f"Output: {output_dir}")
        print(f"{'='*80}")
        print(f"Prompt: {prompt}\n")
        
        generated_files = []
        
        for i in tqdm(range(num_images), desc=f"Generating {stage}"):
            image = self.pipe(
                prompt,
                negative_prompt=self.negative_prompt,
                num_inference_steps=50,
                guidance_scale=7.5,
                height=512,
                width=512
            ).images[0]
            
            filename = f"{stage}_{start_idx + i}.png"
            filepath = os.path.join(output_dir, filename)
            image.save(filepath)
            generated_files.append(filepath)
        
        print(f"\n✓ Generated {num_images} images for {stage}")
        print(f"Files: {stage}_{start_idx}.png to {stage}_{start_idx + num_images - 1}.png")
        return generated_files

print('✓ Generator class loaded!')

In [ ]:
# Initialize generator (this takes 2-3 minutes)
generator = SyntheticDataGenerator()

# Create output directories (Kaggle uses /kaggle/working/)
!mkdir -p /kaggle/working/synthetic_images/CIN1
!mkdir -p /kaggle/working/synthetic_images/CIN2
!mkdir -p /kaggle/working/synthetic_images/CIN3
!mkdir -p /kaggle/working/synthetic_images/Cancer

print('\n✓ Generator initialized with GPU!')
print('Ready to generate images!')

In [ ]:
# Generate 200 images for each minority class
# Total: 800 images, ~2-3 hours on GPU

# CIN1 (200 images, ~30 min)
generator.generate_images(
    stage='CIN1',
    num_images=200,
    output_dir='/kaggle/working/synthetic_images/CIN1',
    start_idx=1
)

# CIN2 (200 images, ~30 min)
generator.generate_images(
    stage='CIN2',
    num_images=200,
    output_dir='/kaggle/working/synthetic_images/CIN2',
    start_idx=1
)

# CIN3 (200 images, ~30 min)
generator.generate_images(
    stage='CIN3',
    num_images=200,
    output_dir='/kaggle/working/synthetic_images/CIN3',
    start_idx=1
)

# Cancer (200 images, ~30 min)
generator.generate_images(
    stage='Cancer',
    num_images=200,
    output_dir='/kaggle/working/synthetic_images/Cancer',
    start_idx=1
)

print('\n' + '='*80)
print('✓ All images generated!')
print('='*80)
print('Summary:')
print(f'  CIN1: 200 images')
print(f'  CIN2: 200 images')
print(f'  CIN3: 200 images')
print(f'  Cancer: 200 images')
print(f'  Total: 800 new images')

In [ ]:
# Verify generated images
import os

for stage in ['CIN1', 'CIN2', 'CIN3', 'Cancer']:
    folder = f'/kaggle/working/synthetic_images/{stage}'
    count = len([f for f in os.listdir(folder) if f.endswith('.png')])
    print(f'{stage}: {count} images')

In [ ]:
# Zip all generated images for download
!cd /kaggle/working && zip -r synthetic_minority_images.zip synthetic_images/

print('✓ Created synthetic_minority_images.zip')
print('File size:')
!ls -lh /kaggle/working/synthetic_minority_images.zip

In [ ]:
# Download instructions for Kaggle
print('✓ Generation complete!')
print('='*80)
print('The zip file is saved at: /kaggle/working/synthetic_minority_images.zip')
print('='*80)
print('\nTo download:')
print('1. Click on "Data" tab in the right sidebar')
print('2. Navigate to "Output" section')
print('3. Click the download button next to synthetic_minority_images.zip')
print('\nThis contains 800 images (200 each for CIN1, CIN2, CIN3, Cancer)')
print('Extract and copy to your training folders!')

---
## Download from Kaggle:

1. **Click "Data" tab** in the right sidebar
2. **Navigate to "Output"** section
3. **Click download** next to `synthetic_minority_images.zip`

## Next Steps on Your Mac:
